# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [10]:
import json
import os
import re
import tempfile
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)

OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

# Voice uses OpenAI Whisper + TTS (same as week 2 day 5). The tutor itself stays on Ollama.
openai_audio = OpenAI() if os.getenv("OPENAI_API_KEY") else None

MODELS = {
    "Llama 3.2": "llama3.2",
    "Gemma 3 270M": "gemma3:270m",
}

In [11]:
EXPERTISE = {
    "Python": "python, software engineering, and debugging. Use short code examples.",
    "LLMs": "large language models, prompting, tokens, and LLM APIs.",
    "Data science": "pandas, statistics, and machine learning fundamentals.",
}


def system_prompt_for(topic):
    focus = EXPERTISE[topic]
    return f"""
You are a helpful technical tutor for Ed Donner's LLM Engineering course.
You answer questions about {focus}
Give a clear, structured explanation in markdown.
If a glossary tool result is available, use it and then add a practical example.
If you do not know, say so.
""".strip()

In [12]:
GLOSSARY = {
    "llm": "A Large Language Model predicts the next token from context. Chat APIs wrap that in a conversation of system/user/assistant messages.",
    "token": "A token is a chunk of text the model reads or writes. Roughly 1 token ≈ 4 characters in English.",
    "prompt": "The text you send the model. A system prompt sets role and tone; a user prompt is the actual request.",
    "tool calling": "The model returns a structured function call instead of (or before) a final answer. Your code runs the function and sends the result back.",
    "rag": "Retrieval-Augmented Generation: look up relevant documents, then include them in the prompt so the model can answer with that context.",
    "embedding": "A numeric vector that represents meaning. Similar texts have similar embeddings, which is how semantic search works.",
    "gradio": "A Python library for building simple UIs (chat, forms, streaming) around ML functions.",
    "ollama": "A local runtime for open-source LLMs. The OpenAI Python client can talk to it at http://localhost:11434/v1.",
}


def lookup_term(term):
    key = term.strip().lower()
    print(f"TOOL called: lookup_term({term})", flush=True)
    meaning = GLOSSARY.get(key)
    if meaning:
        return f"{term}: {meaning}"
    known = ", ".join(sorted(GLOSSARY))
    return f"No glossary entry for '{term}'. Known terms: {known}."


lookup_function = {
    "name": "lookup_term",
    "description": "Look up a short definition of an LLM, Python, or course term such as token, RAG, embedding, Gradio, or Ollama.",
    "parameters": {
        "type": "object",
        "properties": {
            "term": {
                "type": "string",
                "description": "The term to look up, for example 'RAG' or 'token'.",
            },
        },
        "required": ["term"],
        "additionalProperties": False,
    },
}

tools = [{"type": "function", "function": lookup_function}]

In [13]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments or "{}")
        if tool_call.function.name == "lookup_term":
            content = lookup_term(arguments.get("term", ""))
        else:
            content = f"Unknown tool: {tool_call.function.name}"
        responses.append({
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id,
        })
    return responses


def tutor(message, history, model_name, topic):
    model = MODELS[model_name]
    history = [{"role": turn["role"], "content": turn["content"]} for turn in history]
    messages = (
        [{"role": "system", "content": system_prompt_for(topic)}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = ollama.chat.completions.create(
        model=model, messages=messages, tools=tools
    )
    while response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        tool_responses = handle_tool_calls(assistant_message)
        messages.append(assistant_message)
        messages.extend(tool_responses)
        response = ollama.chat.completions.create(
            model=model, messages=messages, tools=tools
        )

    stream = ollama.chat.completions.create(
        model=model, messages=messages, stream=True
    )
    reply = ""
    for chunk in stream:
        reply += chunk.choices[0].delta.content or ""
        yield reply

In [14]:
def for_speech(text):
    text = re.sub(r"```.*?```", " ", text, flags=re.S)
    text = re.sub(r"[#*_>`]", "", text)
    text = re.sub(r"\n+", " ", text).strip()
    return text[:800]


def listener(audio_path):
    if not audio_path:
        return ""
    if openai_audio is None:
        raise gr.Error("Set OPENAI_API_KEY in .env to use the microphone (Whisper).")
    with open(audio_path, "rb") as audio_file:
        return openai_audio.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file,
        ).text


def talker(message):
    if not message:
        return None
    if openai_audio is None:
        raise gr.Error("Set OPENAI_API_KEY in .env to hear spoken replies (TTS).")
    speech = openai_audio.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="onyx",
        input=message,
    )
    path = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3").name
    with open(path, "wb") as f:
        f.write(speech.content)
    return path

In [15]:
def put_message_in_chatbot(message, history):
    return "", history + [{"role": "user", "content": message}]


def stream_and_speak(history, model_name, topic):
    message = history[-1]["content"]
    prior = history[:-1]
    reply = ""
    for partial in tutor(message, prior, model_name, topic):
        reply = partial
        yield prior + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": reply},
        ], None
    yield prior + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": reply},
    ], talker(for_speech(reply))


with gr.Blocks() as ui:
    gr.Markdown("## Course technical tutor (Ollama + voice)")
    gr.Markdown("Answers come from **Llama 3.2**. Speak with the mic (Whisper) and hear the reply (TTS), as in day 5.")
    chatbot = gr.Chatbot(height=420, type="messages")
    audio_output = gr.Audio(label="Spoken reply", autoplay=True)
    with gr.Row():
        model_dropdown = gr.Dropdown(list(MODELS), value="Llama 3.2", label="Ollama model")
        topic_dropdown = gr.Dropdown(list(EXPERTISE), value="LLMs", label="Expertise")
    with gr.Row():
        message = gr.Textbox(label="Type a question", scale=3)
        mic = gr.Audio(sources=["microphone"], type="filepath", label="Or talk", scale=1)

    mic.stop_recording(listener, inputs=mic, outputs=message)
    message.submit(
        put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]
    ).then(
        stream_and_speak,
        inputs=[chatbot, model_dropdown, topic_dropdown],
        outputs=[chatbot, audio_output],
    )

ui.launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


TOOL called: lookup_term(embedding)


Traceback (most recent call last):
  File "d:\learning_projects\core_track\llm_engineering\.venv\Lib\site-packages\gradio\queueing.py", line 849, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\learning_projects\core_track\llm_engineering\.venv\Lib\site-packages\gradio\route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\learning_projects\core_track\llm_engineering\.venv\Lib\site-packages\gradio\blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\learning_projects\core_track\llm_engineering\.venv\Lib\site-packages\gradio\blocks.py", line 1635, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\learning_projects\core_track\llm_engineering\.venv\